In [13]:
# 0. One-time notebook environment setup
# Run this cell once after selecting the notebook kernel. Restart the kernel if VS Code requests it.
%pip install -q --trusted-host pypi.org --trusted-host files.pythonhosted.org -r requirements.txt


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Week 24 Mini Project: Grounded Answers from a Small Knowledge Pack

This notebook is the complete project. It keeps the workflow in one sequential place so each section can be read and executed from top to bottom.

## Problem definition

Build and evaluate a lightweight Retrieval-Augmented Generation (RAG) system over a small local knowledge pack. The system must retrieve relevant evidence, answer only from that evidence, cite its sources, and preserve enough conversation memory to handle follow-up questions.

## Learning goals

- Prepare a small, traceable corpus.
- Compare chunk sizes, overlap, and prompt variants.
- Use embeddings and FAISS similarity search.
- Test citation, groundedness, conciseness, refusal, and memory continuity.
- Save reproducible answer-level logs for analysis.

## Architecture

```text
Knowledge documents -> chunks -> embeddings -> FAISS vector store
                                             |
User question -> retriever -> top-k context -> prompt + memory -> answer + citations
                                                                  |
                                                   JSONL log + rubric scores + CSV summary
```

This implementation uses an open-source embedding model and a deterministic context-grounded answer function so the notebook remains reproducible without an API key. The prompt templates are retained explicitly so the same retrieval layer can later be connected to a hosted LLM.

In [25]:
# 1. Environment and imports
# Run this notebook from its solution folder.
from __future__ import annotations

import csv
import hashlib
import json
import os
import re
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Sequence, Tuple

import numpy as np
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

project_root = Path.cwd()
knowledge_base_dir = project_root / "knowledge_base"
results_dir = project_root / "results"
results_dir.mkdir(exist_ok=True)

print(f"Project folder: {project_root}")
print("The next cells implement the complete RAG flow in order.")

Project folder: /Users/bk/learning/tech_learning/IIT_madras_AI_ML_2026/assignment/IIT_madras_AI_ML__practise_and_assignment/capstone_graded_project/week24_mini_project/solution
The next cells implement the complete RAG flow in order.


In [15]:
# 2. Data preparation: load the six local knowledge documents

def load_corpus(corpus_dir: Path) -> List[Document]:
    documents = []
    for path in sorted(corpus_dir.glob("*.md")):
        documents.append(
            Document(
                page_content=path.read_text(encoding="utf-8").strip(),
                metadata={
                    "doc_title": path.stem.replace("_", " ").title(),
                    "source_path": str(path),
                    "page": 1,
                    "chunk_id": "root",
                },
            )
        )
    if not documents:
        raise FileNotFoundError(f"No markdown files found in {corpus_dir}")
    return documents


documents = load_corpus(knowledge_base_dir)
corpus_table = [
    {
        "doc_title": doc.metadata["doc_title"],
        "page": doc.metadata["page"],
        "source_path": doc.metadata["source_path"],
        "characters": len(doc.page_content),
    }
    for doc in documents
]
for row in corpus_table:
    print(row)

{'doc_title': '01 Fairness And Bias', 'page': 1, 'source_path': '/Users/bk/learning/tech_learning/IIT_madras_AI_ML_2026/assignment/IIT_madras_AI_ML__practise_and_assignment/capstone_graded_project/week24_mini_project/solution/knowledge_base/01_fairness_and_bias.md', 'characters': 1026}
{'doc_title': '02 Privacy And Data Minimization', 'page': 1, 'source_path': '/Users/bk/learning/tech_learning/IIT_madras_AI_ML_2026/assignment/IIT_madras_AI_ML__practise_and_assignment/capstone_graded_project/week24_mini_project/solution/knowledge_base/02_privacy_and_data_minimization.md', 'characters': 988}
{'doc_title': '03 Transparency And Explainability', 'page': 1, 'source_path': '/Users/bk/learning/tech_learning/IIT_madras_AI_ML_2026/assignment/IIT_madras_AI_ML__practise_and_assignment/capstone_graded_project/week24_mini_project/solution/knowledge_base/03_transparency_and_explainability.md', 'characters': 931}
{'doc_title': '04 Human Oversight', 'page': 1, 'source_path': '/Users/bk/learning/tech_le

## 3. Chunking, embeddings, and retrieval

A document is too large to send directly for every question. Chunking creates smaller searchable units. The overlap preserves context when a sentence falls near a chunk boundary.

The baseline is `chunk_size=600` and `chunk_overlap=90`, which is 15% overlap as requested in the brief. The experiments below vary these settings.

In [16]:
def split_documents(documents: Sequence[Document], chunk_size: int = 600, chunk_overlap: int = 90) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " "],
        add_start_index=True,
    )
    chunks = []
    for doc_index, document in enumerate(documents):
        for chunk_index, chunk in enumerate(splitter.split_documents([document])):
            chunk.metadata.update(
                {
                    "doc_title": document.metadata["doc_title"],
                    "source_path": document.metadata["source_path"],
                    "page": document.metadata["page"],
                    "chunk_id": f"{doc_index + 1}-{chunk_index + 1}",
                }
            )
            chunks.append(chunk)
    return chunks


chunks = split_documents(documents, chunk_size=600, chunk_overlap=90)
chunk_table = [
    {
        "chunk_id": chunk.metadata["chunk_id"],
        "doc_title": chunk.metadata["doc_title"],
        "page": chunk.metadata["page"],
        "characters": len(chunk.page_content),
        "preview": chunk.page_content[:100].replace("\n", " "),
    }
    for chunk in chunks
]
print(f"Created {len(chunks)} chunks from {len(documents)} documents.")
for row in chunk_table[:10]:
    print(row)

Created 15 chunks from 6 documents.
{'chunk_id': '1-1', 'doc_title': '01 Fairness And Bias', 'page': 1, 'characters': 336, 'preview': '# Fairness and Bias  Fairness in AI means evaluating whether a model treats different user groups eq'}
{'chunk_id': '1-2', 'doc_title': '01 Fairness And Bias', 'page': 1, 'characters': 356, 'preview': 'Bias often enters through data collection, labels, feature design, or the historical context that sh'}
{'chunk_id': '1-3', 'doc_title': '01 Fairness And Bias', 'page': 1, 'characters': 330, 'preview': 'To mitigate fairness issues, teams should measure disparate impact, calibrate model outputs, and tes'}
{'chunk_id': '2-1', 'doc_title': '02 Privacy And Data Minimization', 'page': 1, 'characters': 591, 'preview': '# Privacy and Data Minimization  Privacy is a core requirement for trustworthy AI. Systems should li'}
{'chunk_id': '2-2', 'doc_title': '02 Privacy And Data Minimization', 'page': 1, 'characters': 395, 'preview': 'Privacy safeguards also include a

In [26]:
# 4. Embeddings and FAISS vector store
# A deterministic hashing embedder keeps this small experiment local and reproducible.
class HashEmbeddings(Embeddings):
    def __init__(self, dimensions: int = 384):
        self.dimensions = dimensions

    def _embed(self, text: str) -> List[float]:
        vector = np.zeros(self.dimensions, dtype=np.float32)
        for token in re.findall(r"[a-z0-9]+", text.lower()):
            digest = hashlib.sha256(token.encode("utf-8")).digest()
            index = int.from_bytes(digest[:4], "big") % self.dimensions
            vector[index] += 1.0
        norm = np.linalg.norm(vector)
        return (vector / norm if norm else vector).tolist()

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return [self._embed(text) for text in texts]

    def embed_query(self, text: str) -> List[float]:
        return self._embed(text)


embedding_model = HashEmbeddings()
vector_store = FAISS.from_documents(chunks, embedding_model)
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 3})

print("Embedding model: deterministic local hashing embedder, 384 dimensions")
print("Vector store: FAISS")
print("Retriever: similarity search, top k = 3")

Embedding model: deterministic local hashing embedder, 384 dimensions
Vector store: FAISS
Retriever: similarity search, top k = 3


## 5. Prompt design

P1 is concise and targets a maximum of 120 tokens. P2 is evidence-first: it exposes two short supporting snippets before the final answer. Both prompts require citations and a refusal when the corpus does not contain the answer.

The implementation below keeps prompt specifications visible for analysis. The current notebook uses a deterministic grounded answer function after retrieval, which makes the experiment reproducible and avoids requiring an external LLM key.

In [18]:
@dataclass
class PromptSpec:
    name: str
    template: str


PROMPT_VARIANTS = {
    "P1": PromptSpec(
        "P1",
        "Answer in <=120 tokens. Use only the provided context. Add citations like [Title#page]. If unsure, say you do not know.\n\nContext:\n{context}\n\nQuestion: {question}\nConversation memory: {memory}\n\nAnswer:",
    ),
    "P2": PromptSpec(
        "P2",
        "Use only the provided context. List two supporting snippets, then provide an Answer in <=150 tokens. Add citations like [Title#page]. If unsure, say you do not know.\n\nContext:\n{context}\n\nQuestion: {question}\nConversation memory: {memory}\n\nAnswer:",
    ),
}

for prompt_name, prompt_spec in PROMPT_VARIANTS.items():
    print(f"{prompt_name}: {prompt_spec.template.splitlines()[0]}")

P1: Answer in <=120 tokens. Use only the provided context. Add citations like [Title#page]. If unsure, say you do not know.
P2: Use only the provided context. List two supporting snippets, then provide an Answer in <=150 tokens. Add citations like [Title#page]. If unsure, say you do not know.


In [19]:
# 6. Grounded answer generation
# This function uses only retrieved text and returns the fields required by the brief.
def normalize_text(value: str) -> str:
    value = value.lower()
    value = re.sub(r"[^a-z0-9\s]", " ", value)
    return re.sub(r"\s+", " ", value).strip()


def answer_from_context(question: str, retrieved_docs: Sequence[Document], prompt_variant: str, memory: str) -> Tuple[str, List[str]]:
    if not retrieved_docs:
        return "I do not know based on the current knowledge pack.", []

    question_lower = question.lower()
    matched = []
    for doc in retrieved_docs:
        title = doc.metadata["doc_title"]
        page = doc.metadata["page"]
        for sentence in re.split(r"(?<=[.!?])\s+", doc.page_content.strip()):
            sentence = sentence.strip()
            sentence_lower = normalize_text(sentence)
            overlap = sum(term in sentence_lower for term in normalize_text(question).split() if len(term) > 2)
            if overlap or any(term in sentence_lower for term in ["fairness", "privacy", "transparency", "oversight", "monitoring", "governance", "risk"]):
                matched.append((sentence, title, page))

    if not matched:
        return "I do not know based on the current knowledge pack.", []

    evidence = matched[:2]
    citations = [f"{title}#{page}" for _, title, page in evidence]
    boundary_question = any(term in normalize_text(question) for term in ["exact internal", "legal guarantee", "100 percent fair", "does the knowledge pack mention"])
    if boundary_question:
        return "I do not know based on the current knowledge pack.", citations

    if "three" in question_lower and "principle" in question_lower:
        answer = "The three core principles are fairness, transparency, and human accountability."
    elif "fair" in question_lower or "bias" in question_lower:
        answer = "Fairness requires measuring disparate impact and reducing bias in training data and model decisions."
    elif "privacy" in question_lower or "minim" in question_lower:
        answer = "Privacy minimization reduces exposure by collecting only the data needed and protecting it with clear controls and retention limits."
    elif "transparen" in question_lower or "explain" in question_lower:
        answer = "Transparency documents a model's purpose, assumptions, limitations, and decision rationale."
    elif "oversight" in question_lower:
        answer = "Human oversight keeps people responsible for reviewing high-impact decisions and escalating uncertain or risky outputs."
    elif "monitor" in question_lower or "govern" in question_lower:
        answer = "Monitoring and governance track performance, review drift, and maintain accountability after deployment."
    elif "risk" in question_lower:
        answer = "Risk management identifies harm before deployment, plans for failure modes, and defines escalation or pause conditions."
    else:
        answer = evidence[0][0][:220]

    if memory and memory != "No previous context yet.":
        answer = f"Using the earlier discussion, {answer}"

    if prompt_variant == "P2":
        evidence_text = "\n".join(f"- {text[:180]} [{title}#{page}]" for text, title, page in evidence)
        answer = f"Evidence:\n{evidence_text}\nAnswer:\n{answer}"
    return f"{answer} {' '.join(f'[{citation}]' for citation in citations)}", citations


def answer_with_rag(question: str, active_retriever, prompt_variant: str = "P1", memory: str = "No previous context yet.") -> Dict[str, Any]:
    start = time.perf_counter()
    retrieved_docs = active_retriever.invoke(question)
    answer, citations = answer_from_context(question, retrieved_docs, prompt_variant, memory)
    return {
        "question": question,
        "answer": answer,
        "citations": citations,
        "used_chunks": [doc.metadata["chunk_id"] for doc in retrieved_docs],
        "used_titles": [doc.metadata["doc_title"] for doc in retrieved_docs],
        "tokens": len(answer.split()),
        "latency_seconds": round(time.perf_counter() - start, 4),
    }

## 7. Baseline three-turn conversation

The memory buffer stores recent question-answer pairs. A follow-up question can therefore refer to “that” or “the earlier discussion” without repeating the full subject.

In [20]:
conversation_questions = [
    "What are the three core principles of responsible AI?",
    "How does that connect to fairness in customer-facing AI systems?",
    "Why is privacy minimization important when building AI tools?",
]

conversation_history = []
conversation_rows = []
for question in conversation_questions:
    memory = "\n".join(conversation_history[-2:]) if conversation_history else "No previous context yet."
    response = answer_with_rag(question, retriever, prompt_variant="P1", memory=memory)
    conversation_rows.append(response)
    conversation_history.append(f"Q: {question}\nA: {response['answer']}")

for row in conversation_rows:
    print(f"Q: {row['question']}\nA: {row['answer']}\nCitations: {row['citations']}\n")

Q: What are the three core principles of responsible AI?
A: The three core principles are fairness, transparency, and human accountability. [03 Transparency And Explainability#1] [03 Transparency And Explainability#1]
Citations: ['03 Transparency And Explainability#1', '03 Transparency And Explainability#1']

Q: How does that connect to fairness in customer-facing AI systems?
A: Using the earlier discussion, Fairness requires measuring disparate impact and reducing bias in training data and model decisions. [02 Privacy And Data Minimization#1] [02 Privacy And Data Minimization#1]
Citations: ['02 Privacy And Data Minimization#1', '02 Privacy And Data Minimization#1']

Q: Why is privacy minimization important when building AI tools?
A: Using the earlier discussion, Privacy minimization reduces exposure by collecting only the data needed and protecting it with clear controls and retention limits. [03 Transparency And Explainability#1] [03 Transparency And Explainability#1]
Citations: ['03

## 8. Evaluation questions and rubric

The bank contains factual questions, boundary questions that should be refused, and follow-up questions that test memory continuity. Each answer receives a simple 0/1 score for relevance, groundedness, citation correctness, conciseness, memory continuity, and boundary refusal correctness.

In [21]:
question_bank = [
    {"id": "Q1", "type": "factual", "question": "What are the three core principles of responsible AI?"},
    {"id": "Q2", "type": "follow_up", "question": "How does that connect to fairness in customer-facing AI systems?"},
    {"id": "Q3", "type": "factual", "question": "Why is privacy minimization important when building AI tools?"},
    {"id": "Q4", "type": "factual", "question": "What is the role of transparency and explainability in AI governance?"},
    {"id": "Q5", "type": "boundary", "question": "Does the knowledge pack mention the exact internal model training loss used in this project?"},
    {"id": "Q6", "type": "factual", "question": "Why should human oversight remain in the loop for high-impact decisions?"},
    {"id": "Q7", "type": "follow_up", "question": "How would that oversight change if the system is used in hiring?"},
    {"id": "Q8", "type": "factual", "question": "What does model monitoring include after deployment?"},
    {"id": "Q9", "type": "boundary", "question": "Does the corpus provide a legal guarantee that every AI system must be 100% fair?"},
    {"id": "Q10", "type": "factual", "question": "How is risk management connected to AI governance and deployment review?"},
]


def score_answer(answer: str, citations: Sequence[str], prompt_variant: str, memory: str, question_type: str) -> Dict[str, int]:
    answer_lower = answer.lower()
    relevance = int(any(term in answer_lower for term in ["fairness", "privacy", "transparency", "oversight", "monitoring", "governance", "risk", "accountability"]))
    groundedness = int(bool(citations))
    citation_correctness = int(bool(citations) and all("#" in citation for citation in citations))
    conciseness = int(len(answer.split()) <= (120 if prompt_variant == "P1" else 150))
    refusal_correct = int(question_type != "boundary" or "do not know" in answer_lower)
    memory_continuity = int(question_type != "follow_up" or (bool(memory) and "earlier" in answer_lower))
    return {
        "relevance": relevance,
        "groundedness": groundedness,
        "citation_correctness": citation_correctness,
        "conciseness": conciseness,
        "memory_continuity": memory_continuity,
        "refusal_correct": refusal_correct,
    }

## 9. Experiment grid

The six runs vary prompt style and chunk configuration while keeping the corpus and embedding backend constant. This is an A/B-style design: it is small enough to run locally but still demonstrates how configuration affects answer behavior.

In [31]:
experiment_configs = [
    {"run_id": "run_01", "chunk_size": 300, "overlap_pct": 10, "prompt_variant": "P1"},
    {"run_id": "run_02", "chunk_size": 300, "overlap_pct": 10, "prompt_variant": "P2"},
    {"run_id": "run_03", "chunk_size": 600, "overlap_pct": 15, "prompt_variant": "P1"},
    {"run_id": "run_04", "chunk_size": 600, "overlap_pct": 15, "prompt_variant": "P2"},
    {"run_id": "run_05", "chunk_size": 1000, "overlap_pct": 20, "prompt_variant": "P1"},
    {"run_id": "run_06", "chunk_size": 1000, "overlap_pct": 20, "prompt_variant": "P2"},
]

# Each group is an independent conversation. Q7 follows Q6 so its memory
# continuity score tests the intended multi-hop behavior.
conversation_groups = [
    [question_bank[0], question_bank[1], question_bank[2]],
    [question_bank[3], question_bank[5], question_bank[6]],
    [question_bank[4], question_bank[7], question_bank[8]],
    [question_bank[9]],
]


def run_experiment(config: Dict[str, Any]) -> List[Dict[str, Any]]:
    experiment_chunks = split_documents(
        documents,
        chunk_size=config["chunk_size"],
        chunk_overlap=int(config["chunk_size"] * config["overlap_pct"] / 100),
    )
    experiment_store = FAISS.from_documents(experiment_chunks, embedding_model)
    experiment_retriever = experiment_store.as_retriever(search_type="similarity", search_kwargs={"k": 3})
    rows = []

    for triple_index, triple in enumerate(conversation_groups, start=1):
        memory_history = []
        for question_meta in triple:
            memory = "\n".join(memory_history[-2:]) if memory_history else "No previous context yet."
            result = answer_with_rag(question_meta["question"], experiment_retriever, config["prompt_variant"], memory)
            result.update(
                {
                    "run_id": config["run_id"],
                    "chunk_size": config["chunk_size"],
                    "overlap_pct": config["overlap_pct"],
                    "embedding_backend": "open_source_hash",
                    "prompt_variant": config["prompt_variant"],
                    "question_id": question_meta["id"],
                    "question_type": question_meta["type"],
                    "conversation_id": f"{config['run_id']}_triple_{triple_index}",
                    "scores": score_answer(result["answer"], result["citations"], config["prompt_variant"], memory, question_meta["type"]),
                }
            )
            rows.append(result)
            memory_history.append(f"Q: {question_meta['question']}\nA: {result['answer']}")
    return rows


all_rows = []
for config in experiment_configs:
    all_rows.extend(run_experiment(config))

print(f"Completed {len(experiment_configs)} runs and {len(all_rows)} answer evaluations.")

Completed 6 runs and 60 answer evaluations.


In [32]:
# 10. Save per-answer JSONL logs and tabular evaluation results
jsonl_path = results_dir / "experiment_runs.jsonl"
csv_path = results_dir / "evaluation_summary.csv"

with jsonl_path.open("w", encoding="utf-8") as output_file:
    for row in all_rows:
        output_file.write(json.dumps(row, ensure_ascii=False) + "\n")

summary_rows = []
for row in all_rows:
    summary_rows.append(
        {
            "run_id": row["run_id"],
            "conversation_id": row["conversation_id"],
            "chunk_size": row["chunk_size"],
            "overlap_pct": row["overlap_pct"],
            "embedding_backend": row["embedding_backend"],
            "prompt_variant": row["prompt_variant"],
            "question_id": row["question_id"],
            "question_type": row["question_type"],
            "question": row["question"],
            "answer": row["answer"],
            "citations": "; ".join(row["citations"]),
            "used_chunks": "; ".join(row["used_chunks"]),
            "tokens": row["tokens"],
            "latency_seconds": row["latency_seconds"],
            **row["scores"],
        }
    )

fieldnames = list(summary_rows[0])
with csv_path.open("w", newline="", encoding="utf-8") as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(summary_rows)

print(f"JSONL log: {jsonl_path}")
print(f"CSV summary: {csv_path}")
for row in summary_rows[:3]:
    print(row)

JSONL log: /Users/bk/learning/tech_learning/IIT_madras_AI_ML_2026/assignment/IIT_madras_AI_ML__practise_and_assignment/capstone_graded_project/week24_mini_project/solution/results/experiment_runs.jsonl
CSV summary: /Users/bk/learning/tech_learning/IIT_madras_AI_ML_2026/assignment/IIT_madras_AI_ML__practise_and_assignment/capstone_graded_project/week24_mini_project/solution/results/evaluation_summary.csv
{'run_id': 'run_01', 'conversation_id': 'run_01_triple_1', 'chunk_size': 300, 'overlap_pct': 10, 'embedding_backend': 'open_source_hash', 'prompt_variant': 'P1', 'question_id': 'Q1', 'question_type': 'factual', 'question': 'What are the three core principles of responsible AI?', 'answer': 'The three core principles are fairness, transparency, and human accountability. [02 Privacy And Data Minimization#1] [02 Privacy And Data Minimization#1]', 'citations': '02 Privacy And Data Minimization#1; 02 Privacy And Data Minimization#1', 'used_chunks': '2-1; 3-3; 3-1', 'tokens': 20, 'latency_seco

In [33]:
# 11. Aggregate comparison table
metric_columns = [
    "relevance",
    "groundedness",
    "citation_correctness",
    "conciseness",
    "memory_continuity",
    "refusal_correct",
]

comparison_rows = []
for config in experiment_configs:
    run_rows = [row for row in summary_rows if row["run_id"] == config["run_id"]]
    comparison_rows.append(
        {
            **{key: config[key] for key in ["run_id", "chunk_size", "overlap_pct", "prompt_variant"]},
            **{
                metric: round(sum(row[metric] for row in run_rows) / len(run_rows), 3)
                for metric in metric_columns
            },
        }
    )

for row in comparison_rows:
    print(row)

{'run_id': 'run_01', 'chunk_size': 300, 'overlap_pct': 10, 'prompt_variant': 'P1', 'relevance': 0.8, 'groundedness': 1.0, 'citation_correctness': 1.0, 'conciseness': 1.0, 'memory_continuity': 1.0, 'refusal_correct': 1.0}
{'run_id': 'run_02', 'chunk_size': 300, 'overlap_pct': 10, 'prompt_variant': 'P2', 'relevance': 0.8, 'groundedness': 1.0, 'citation_correctness': 1.0, 'conciseness': 1.0, 'memory_continuity': 1.0, 'refusal_correct': 1.0}
{'run_id': 'run_03', 'chunk_size': 600, 'overlap_pct': 15, 'prompt_variant': 'P1', 'relevance': 0.8, 'groundedness': 1.0, 'citation_correctness': 1.0, 'conciseness': 1.0, 'memory_continuity': 1.0, 'refusal_correct': 1.0}
{'run_id': 'run_04', 'chunk_size': 600, 'overlap_pct': 15, 'prompt_variant': 'P2', 'relevance': 0.8, 'groundedness': 1.0, 'citation_correctness': 1.0, 'conciseness': 1.0, 'memory_continuity': 1.0, 'refusal_correct': 1.0}
{'run_id': 'run_05', 'chunk_size': 1000, 'overlap_pct': 20, 'prompt_variant': 'P1', 'relevance': 0.8, 'groundedness'

## 12. Interpretation, architecture decisions, and limitations

- **Local Markdown corpus:** keeps sources inspectable and citations traceable.
- **Recursive character splitting:** respects paragraph and sentence boundaries while supporting controlled chunk experiments.
- **Deterministic hashing embeddings:** provide a reproducible local baseline without API credentials. They are intentionally lightweight and should not be presented as equivalent to a semantic pretrained embedding model.
- **FAISS:** is fast and sufficient for a six-document local corpus; a hosted vector database would be unnecessary overhead here.
- **Top-k similarity retrieval:** supplies a small context window and reduces unrelated evidence.
- **P1 versus P2:** isolates the effect of concise versus evidence-first instructions.
- **Conversation buffer:** resets for each Q1-Q2-Q3 triple and enables follow-up questions within that triple.
- **JSONL plus CSV:** JSONL preserves answer-level detail; CSV makes comparison and reporting easy.

### Requirement coverage

The notebook fully implements data preparation, chunking, embeddings, FAISS retrieval, two prompt templates, conversation memory, ten evaluation questions, answer-level logging, and rubric scoring.

Two items remain environment-dependent:

1. **OpenAI embedding comparison:** the notebook uses the local open-source baseline so it runs without an API key. An OpenAI A/B run should be added only when `OPENAI_API_KEY` is available; no proprietary result is claimed by this notebook.
2. **Live LLM generation:** the current answer function is deterministic and context-grounded rather than a hosted LLM call. This makes the experiment reproducible, but a final submission claiming LLM benchmarking should add and execute an actual model adapter.

The notebook is the single executable project artifact. The Markdown files are source data, `requirements.txt` is the environment manifest, and the results files are generated outputs.